1. Определение формата ссылки для пагинации

In [1]:
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://lifehacker.ru"
TOPIC_URL = f"{BASE_URL}/topics/technology/"
PAGES = 10
DELAY = 0.4          # пауза между запросами, чтобы не создавать нагрузку на сайт

# Сайт отдаёт контент только «браузерам», поэтому подставляем User-Agent.
# Session переиспользует TCP-соединение — 310 запросов проходят заметно быстрее.
session = requests.Session()
session.headers.update({
    "User-Agent": ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36"),
    "Accept-Language": "ru-RU,ru;q=0.9",
})


def fetch(url: str, attempts: int = 3) -> str | None:
    """Скачивает страницу, повторяя попытку при сетевой ошибке.

    Возвращает None, если страница так и не открылась: одна недоступная статья
    не должна ронять сбор остальных трёхсот.
    """
    for attempt in range(1, attempts + 1):
        try:
            response = session.get(url, timeout=30)
            if response.status_code == 200:
                return response.text
            print(f"    {url} -> статус {response.status_code}")
            return None
        except requests.RequestException as error:
            print(f"    попытка {attempt}/{attempts} для {url}: {type(error).__name__}")
            time.sleep(2 * attempt)
    return None


print("страница пагинации:", f"{TOPIC_URL}?page=2")

страница пагинации: https://lifehacker.ru/topics/technology/?page=2


2. Определение уникальных классов блока с названием и содержанием материала

В разметке списка каждая карточка материала — это div.lh-small-article-card, внутри неё есть ссылка a.lh-small-article-card__link. Сам заголовок в ссылке отсутствует (он находится в соседнем блоке __title), дополнительно продублирован в атрибуте title, поэтому ссылку и предварительный заголовок можно взять из одного элемента.

На странице материала заголовок лежит в h1, а текст — в article.single-article__post-content.

In [2]:
CARD_LINK = "a.lh-small-article-card__link[href]"
ARTICLE_BODY = "article.single-article__post-content"


def extract_links(page_html: str) -> list[str]:
    """Достаёт со страницы списка абсолютные ссылки на материалы."""
    soup = BeautifulSoup(page_html, "lxml")
    links = []
    for tag in soup.select(CARD_LINK):
        href = tag["href"]
        # отсекаем ссылки на рубрики и внешние домены — нужны только материалы
        if href.startswith("/") and not href.startswith("/topics/"):
            links.append(urljoin(BASE_URL, href))
    return links


# Проверяем на первой странице, что селектор рабочий
first_page = fetch(TOPIC_URL)
sample_links = extract_links(first_page)
print(f"ссылок на первой странице: {len(sample_links)}")
sample_links[:3]

ссылок на первой странице: 30


['https://lifehacker.ru/v-minecraft-vossozdali-shveicariyu/',
 'https://lifehacker.ru/zvonok-na-112-bez-sim-karty/',
 'https://lifehacker.ru/samsung-razrabatyvaet-skladnoi-planshet/']

3. Получение ссылок с десяти страниц списка

In [4]:
article_links: list[str] = []
seen: set[str] = set()

for page in range(1, PAGES + 1):
    url = TOPIC_URL if page == 1 else f"{TOPIC_URL}?page={page}"
    html = fetch(url)
    if html is None:
        print(f"страница {page}: не получена, пропускаем")
        continue

    found = extract_links(html)
    # материалы могут повторяться между страницами, если список обновился
    # прямо во время сбора, поэтому храним множество уже виденных ссылок
    new = [link for link in found if link not in seen]
    seen.update(new)
    article_links.extend(new)

    print(f"страница {page:>2}: найдено {len(found):>3}, новых {len(new):>3}, "
          f"всего {len(article_links):>3}")
    time.sleep(DELAY)

print(f"\nВсего уникальных ссылок: {len(article_links)}")

страница  1: найдено  30, новых  30, всего  30
страница  2: найдено  30, новых  30, всего  60
страница  3: найдено  30, новых  30, всего  90
страница  4: найдено  30, новых  30, всего 120
страница  5: найдено  30, новых  30, всего 150
страница  6: найдено  30, новых  30, всего 180
страница  7: найдено  30, новых  30, всего 210
страница  8: найдено  30, новых  30, всего 240
страница  9: найдено  30, новых  30, всего 270
страница 10: найдено  30, новых  30, всего 300

Всего уникальных ссылок: 300


4. Скачивание и разбор каждого материала

In [5]:
def parse_article(html: str) -> tuple[str, str]:
    """Возвращает заголовок и текст материала.

    Сбор текста производим не целиком из контейнера, а по абзацам и подзаголовкам
    """
    soup = BeautifulSoup(html, "lxml")

    heading = soup.find("h1")
    title = heading.get_text(" ", strip=True) if heading else ""

    body = soup.select_one(ARTICLE_BODY)
    if body is None:
        return title, ""

    parts = [tag.get_text(" ", strip=True) for tag in body.find_all(["p", "h2", "h3"])]
    text = "\n".join(part for part in parts if part)
    return title, text


records = []
failed = []

for number, link in enumerate(article_links, start=1):
    html = fetch(link)
    if html is None:
        failed.append(link)
        continue

    title, text = parse_article(html)
    records.append({"url": link, "title": title, "text": text,
                    "text_length": len(text)})

    if number % 25 == 0 or number == len(article_links):
        print(f"обработано {number:>3} из {len(article_links)}")
    time.sleep(DELAY)

print(f"\nВсего собрано материалов: {len(records)}, не удалось: {len(failed)}")

обработано  25 из 300
обработано  50 из 300
обработано  75 из 300
обработано 100 из 300
обработано 125 из 300
обработано 150 из 300
обработано 175 из 300
обработано 200 из 300
обработано 225 из 300
обработано 250 из 300
обработано 275 из 300
обработано 300 из 300

Всего собрано материалов: 300, не удалось: 0


5. Создание датайфрейма с полученными данными

In [6]:
articles = pd.DataFrame(records)
print(f"строк: {articles.shape[0]}, столбцов: {articles.shape[1]}")
articles.head()

строк: 300, столбцов: 4


,url,title,text,text_length
0,https://lifehacker.ru/v-minecraft-vossozdali-s...,22-летний студент воссоздал в Minecraft всю Шв...,22-летний студент Высшей технической школы Цюр...,1586
1,https://lifehacker.ru/zvonok-na-112-bez-sim-ka...,Объясняем за минуту: как звонок по номеру 112 ...,"112 — единый номер экстренных служб, который п...",2476
2,https://lifehacker.ru/samsung-razrabatyvaet-sk...,Samsung разрабатывает гигантский складной план...,Samsung начала разработку крупного складного п...,1005
3,https://lifehacker.ru/baseus-i-ugreen-upadut-v...,"10 аксессуаров от Baseus и Ugreen, которые упа...",Уже 1 ноября стартует главная распродажа года ...,6125
4,https://lifehacker.ru/samyj-avtonomnyj-v-mire-...,Самый автономный в мире смартфон отдают со ски...,В рейтинге автономности смартфонов авторитетно...,2316


In [7]:
# Проверка собранного материала
print(f"Пустые заголовки: {(articles['title'] == '').sum()}")
print(f"Пустой текст:    {(articles['text'] == '').sum()}")
print(f"Дубликатов по url: {articles['url'].duplicated().sum()}")
print()
print(articles["text_length"].describe().round(0).to_string())

Пустые заголовки: 0
Пустой текст:    0
Дубликатов по url: 0

count      300.0
mean      3779.0
std       3598.0
min        561.0
25%       1516.0
50%       2053.0
75%       5036.0
max      23419.0


In [10]:
articles.to_csv("technology.csv", index=False)
print("сохранено в technology.csv")
print(f"размер файла: {round(len(articles.to_csv(index=False).encode()) / 1024)} КБ")

сохранено в technology.csv
размер файла: 2038 КБ
